In [ ]:
!pip install music21 tensorflow

In [ ]:
from music21 import converter, instrument, note, chord, stream
import glob
import numpy as np
import random

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.utils import to_categorical


In [ ]:
notes = []

# Read all MIDI files
for file in glob.glob("/content/*.mid*"):

    print("Loading:", file)

    midi = converter.parse(file)

    notes_to_parse = midi.flatten().notes

    for element in notes_to_parse:

        # Single notes
        if isinstance(element, note.Note):
            notes.append(str(element.pitch))

        # Chords
        elif isinstance(element, chord.Chord):
            notes.append('.'.join(str(n) for n in element.normalOrder))

print("\nTotal Notes Extracted:", len(notes))
print(notes[:20])


Loading: /content/music11.mid
Loading: /content/music23.mid
Loading: /content/music15.mid
Loading: /content/music16.mid
Loading: /content/music4.mid
Loading: /content/music25.mid
Loading: /content/music3.mid
Loading: /content/music9.mid
Loading: /content/music2.mid
Loading: /content/music1.mid
Loading: /content/music6.mid
Loading: /content/music18.mid
Loading: /content/music27.mid
Loading: /content/music19.mid
Loading: /content/music13.mid
Loading: /content/music22.mid
Loading: /content/music8.mid
Loading: /content/music10.mid
Loading: /content/music5.mid
Loading: /content/music7.mid
Loading: /content/music30.mid
Loading: /content/music17.mid
Loading: /content/music26.mid
Loading: /content/music12.mid
Loading: /content/music28.mid
Loading: /content/music29.mid
Loading: /content/music14.mid
Loading: /content/music24.mid
Loading: /content/music20.mid
Loading: /content/music21.mid

Total Notes Extracted: 50576
['E5', 'E-5', 'E5', 'E-5', 'E5', 'B4', 'D5', 'C5', 'A4', 'A2', 'E3', 'A3', 'C4'

In [ ]:
# Unique notes
pitchnames = sorted(set(notes))

# Convert notes to integers
note_to_int = dict((note, number) for number, note in enumerate(pitchnames))

sequence_length = 50

network_input = []
network_output = []

for i in range(len(notes) - sequence_length):

    sequence_in = notes[i:i + sequence_length]
    sequence_out = notes[i + sequence_length]

    network_input.append([note_to_int[char] for char in sequence_in])
    network_output.append(note_to_int[sequence_out])

n_patterns = len(network_input)

# Reshape input
network_input = np.reshape(
    network_input,
    (n_patterns, sequence_length, 1)
)

# Normalize
network_input = network_input / float(len(pitchnames))

# One-hot encode output
network_output = to_categorical(network_output)

print("Total Patterns:", n_patterns)
print("Unique Notes:", len(pitchnames))

Total Patterns: 50526
Unique Notes: 356


In [ ]:
model = Sequential()

model.add(LSTM(
    256,
    input_shape=(network_input.shape[1], network_input.shape[2]),
    return_sequences=True
))

model.add(Dropout(0.3))

model.add(LSTM(256))

model.add(Dense(256))

model.add(Dropout(0.3))

model.add(Dense(len(pitchnames), activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam'
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50, 256)        │       264,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 356)            │        91,492 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 946,788 (3.61 MB)

 Trainable params: 946,788 (3.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(
    network_input,
    network_output,
    epochs=50,
    batch_size=64
)

Epoch 1/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 14s 12ms/step - loss: 4.7714
Epoch 2/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - loss: 4.5793
Epoch 3/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - loss: 4.5542
Epoch 4/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - loss: 4.5326
Epoch 5/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - loss: 4.5166
Epoch 6/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - loss: 4.4963
Epoch 7/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - loss: 4.4740
Epoch 8/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - loss: 4.4379
Epoch 9/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - loss: 4.3850
Epoch 10/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - loss: 4.3255
Epoch 11/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - loss: 4.2599
Epoch 12/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - loss: 4.1745
Epoch 13/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - loss: 4.0646
Epoch 14/50
790/790 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - loss: 3.9422
Epoch 15/50
790/790 ━━━━━━━━━

In [ ]:
start = random.randint(0, len(network_input)-1)

pattern = network_input[start]

prediction_output = []

int_to_note = dict(
    (number, note) for number, note in enumerate(pitchnames)
)

# Generate 200 notes
for note_index in range(200):

    prediction_input = np.reshape(pattern, (1, len(pattern), 1))
    prediction_input = prediction_input / float(len(pitchnames))

    prediction = model.predict(prediction_input, verbose=0)

    index = np.argmax(prediction)

    result = int_to_note[index]

    prediction_output.append(result)

    pattern = np.append(pattern, index)
    pattern = pattern[1:]

print(prediction_output[:20])


['1.4.7.9', '1.4.7.9', '1.4.7.9', '1.4.7.9', '1.4.7.9', '1.4.7.9', '1.4.7.9', '1.4.7.9', '10.1.4', '10.1.4', '10.1.4', '10.1', '10.1', '10.1', '10.1', '10.1', '10.1', '10.1', '10.1', '10.1']


In [ ]:
offset = 0
output_notes = []

for pattern in prediction_output:

    # Chord
    if ('.' in pattern) or pattern.isdigit():

        notes_in_chord = pattern.split('.')
        notes = []

        for current_note in notes_in_chord:

            new_note = note.Note(int(current_note))
            new_note.storedInstrument = instrument.Piano()
            notes.append(new_note)

        new_chord = chord.Chord(notes)
        new_chord.offset = offset
        output_notes.append(new_chord)

    # Single Note
    else:

        new_note = note.Note(pattern)
        new_note.offset = offset
        new_note.storedInstrument = instrument.Piano()
        output_notes.append(new_note)

    offset += 0.5

midi_stream = stream.Stream(output_notes)

midi_stream.write('midi', fp='generated_music.mid')

print("Music Generated Successfully!")

Music Generated Successfully!


In [ ]:
from google.colab import files

files.download('generated_music.mid')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
model.save("music_model.h5")

In [13]:
from google.colab import files
files.download("music_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>